In [19]:
import numpy as np
from collections import Counter

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler


In [20]:
data = load_breast_cancer()
X = data.data
y = data.target


In [21]:
scaler = StandardScaler()
X = scaler.fit_transform(X)


In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [28]:
def euclidean_distance(a, b):
    return np.sqrt(np.sum((a - b) ** 2)) 

class KNN:
    def __init__(self, k=5):
        self.k = k

    def fit(self, X, y):
        self.X_train = X
        self.y_train = y

    def predict(self, X):
        predictions = [self._predict(x) for x in X ]
        return np.array(predictions)
    
    def _predict(self, x):
        distance = [euclidean_distance(x, x_train) for x_train in self.X_train]
        k_indices = np.argsort(distance)[:self.k]
        k_nearest_labels = [self.y_train[i] for i in k_indices]
        most_common = Counter(k_nearest_labels).most_common(1)
        return most_common[0][0]

In [29]:
class SVM:
    def __init__(self, lr=0.001, lambda_param=0.01, n_iters=1000):
        self.lr = lr
        self.lambda_param = lambda_param
        self.n_iters = n_iters

    def fit(self, X, y):
        y = np.where(y == 0, -1, 1)

        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0

        for _ in range(self.n_iters):
            for idx, x_i in enumerate(X):
                condition = y[idx] * (np.dot(x_i, self.w) - self.b) >= 1
                if condition:
                    self.w -= self.lr * (2 * self.lambda_param * self.w)
                else:
                    self.w -= self.lr * (2 * self.lambda_param * self.w - y[idx] * x_i)
                    self.b -= self.lr * y[idx]

    def predict(self, X):
        linear_output = np.dot(X, self.w) - self.b
        return np.where(linear_output >= 0, 1, 0)


In [30]:
knn = KNN(k=7)
knn.fit(X_train, y_train)

svm = SVM()
svm.fit(X_train, y_train)


In [31]:
knn_pred = knn.predict(X_test)
svm_pred = svm.predict(X_test)


In [32]:
print("KNN Accuracy :", accuracy_score(y_test, knn_pred))
print("SVM Accuracy :", accuracy_score(y_test, svm_pred))

print("\nKNN Confusion Matrix:\n", confusion_matrix(y_test, knn_pred))
print("\nSVM Confusion Matrix:\n", confusion_matrix(y_test, svm_pred))


KNN Accuracy : 0.9473684210526315
SVM Accuracy : 0.9824561403508771

KNN Confusion Matrix:
 [[40  3]
 [ 3 68]]

SVM Confusion Matrix:
 [[41  2]
 [ 0 71]]
